# Part 1: Data Extraction

This notebook extracts raw data from PostgreSQL, MongoDB, and the Open-Meteo weather API.

In [14]:
import os
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests

from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url
from pymongo import MongoClient

print("All libraries imported successfully!")

All libraries imported successfully!


In [15]:
# Find the project root automatically
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    PROJECT_DIR = current_dir
else:
    PROJECT_DIR = current_dir.parent

# Define data folders
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

# Create raw folder if it doesn't exist
RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)

Project directory: d:\Atomcamp AI Bootcamp\6. ETL Module\Online-Shop-ETL-Project
Raw data directory: d:\Atomcamp AI Bootcamp\6. ETL Module\Online-Shop-ETL-Project\data\raw


In [10]:
import os
from dotenv import load_dotenv
from pathlib import Path

PROJECT_DIR = Path.cwd().parent

load_dotenv(PROJECT_DIR / ".env", override=True)

PG_URL = os.getenv("PG_URL")
MONGO_URL = os.getenv("MONGO_URL")
RO_PASSWORD = os.getenv("RO_PASSWORD")

print("Neon URL loaded:", bool(PG_URL))
print("MongoDB URL loaded:", bool(MONGO_URL))
print("Read-only password loaded:", bool(RO_PASSWORD))

if PG_URL:
    print("PostgreSQL scheme:", PG_URL.split("://")[0])

if MONGO_URL:
    print("MongoDB scheme:", MONGO_URL.split("://")[0])

Neon URL loaded: True
MongoDB URL loaded: True
Read-only password loaded: True
PostgreSQL scheme: postgresql
MongoDB scheme: mongodb+srv


In [16]:
# Load credentials from the project root .env file
load_dotenv(PROJECT_DIR / ".env", override=True)

PG_URL = os.getenv("PG_URL")
MONGO_URL = os.getenv("MONGO_URL")

if not PG_URL:
    raise ValueError("PG_URL is missing from .env")

if not MONGO_URL:
    raise ValueError("MONGO_URL is missing from .env")

print("Neon credentials loaded:", bool(PG_URL))
print("MongoDB credentials loaded:", bool(MONGO_URL))

Neon credentials loaded: True
MongoDB credentials loaded: True


In [17]:
# Configure PostgreSQL connection
parsed_url = make_url(PG_URL.strip())

# Use psycopg2 as the database driver
parsed_url = parsed_url.set(
    drivername="postgresql+psycopg2"
)

engine = create_engine(
    parsed_url,
    pool_pre_ping=True,
    connect_args={"connect_timeout": 15}
)

# Test the connection
with engine.connect() as connection:
    database_name = connection.execute(
        text("SELECT current_database()")
    ).scalar()

print("Connected to PostgreSQL!")
print("Database:", database_name)

Connected to PostgreSQL!
Database: neondb


In [18]:
client = MongoClient(
    MONGO_URL,
    serverSelectionTimeoutMS=15000
)

# Test connection
client.admin.command("ping")

# Select database and collection
mongo_db = client["shop"]
mongo_collection = mongo_db["orders"]

print("Connected to MongoDB Atlas!")
print("Database: shop")
print("Collection: orders")

Connected to MongoDB Atlas!
Database: shop
Collection: orders


In [19]:
# Check the tables available in PostgreSQL
with engine.connect() as connection:

    tables = pd.read_sql(
        text("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name
        """),
        connection
    )

display(tables)

,table_name
0,customers
1,orders


In [20]:
# check the number of records

with engine.connect() as connection:

    customers_count = connection.execute(
        text("SELECT COUNT(*) FROM customers")
    ).scalar()

    orders_count = connection.execute(
        text("SELECT COUNT(*) FROM orders")
    ).scalar()

print("Customers:", customers_count)
print("Orders:", orders_count)

Customers: 2000
Orders: 200000


In [ ]:
# Extract customers from PostgreSQL

with engine.connect() as connection:

    customers_df = pd.read_sql(
        text("SELECT * FROM customers"),
        connection
    )

# Save raw customer data
customers_df.to_csv(
    RAW_DIR / "postgres_customers.csv",
    index=False
)

print("Customers extracted:", len(customers_df))

display(customers_df.head())

#Retrieves all customer records and saves them as a CSV file.

Customers extracted: 2000


,customer_id,name,city,segment,signup_date
0,1,Usman Sheikh,Karachi,Consumer,2022-11-28
1,2,Iqra Khan,Faisalabad,Consumer,2022-11-11
2,3,Maryam Raza,Faisalabad,Consumer,2023-10-20
3,4,Ali Sheikh,Lahore,Consumer,2022-08-09
4,5,Hamza Qureshi,Karachi,Consumer,2022-11-10


In [22]:
# Extract PostgreSQL orders in chunks

orders_file = RAW_DIR / "postgres_orders.csv"

# Remove an older extraction file if present
if orders_file.exists():
    orders_file.unlink()

total_rows = 0

with engine.connect() as connection:

    chunks = pd.read_sql_query(
        text("SELECT * FROM orders ORDER BY order_id"),
        connection,
        chunksize=50000
    )

    for i, chunk in enumerate(chunks):

        # Write the first chunk with headers
        # Append subsequent chunks without repeating headers
        chunk.to_csv(
            orders_file,
            mode="w" if i == 0 else "a",
            header=(i == 0),
            index=False
        )

        total_rows += len(chunk)

        print(
            f"Extracted chunk {i + 1}: "
            f"{len(chunk):,} rows"
        )

print("Total PostgreSQL orders extracted:", total_rows)

Extracted chunk 1: 50,000 rows
Extracted chunk 2: 50,000 rows
Extracted chunk 3: 50,000 rows
Extracted chunk 4: 50,000 rows
Total PostgreSQL orders extracted: 200000


In [23]:
# Extract incremental PostgreSQL orders

last_run = "2026-10-01"

incremental_query = text("""
    SELECT *
    FROM orders
    WHERE updated_at >= :last_run
    ORDER BY updated_at
""")

with engine.connect() as connection:

    incremental_orders_df = pd.read_sql(
        incremental_query,
        connection,
        params={"last_run": last_run}
    )

incremental_orders_df.to_csv(
    RAW_DIR / "postgres_incremental_orders.csv",
    index=False
)

print(
    "Incremental orders extracted:",
    len(incremental_orders_df)
)

display(incremental_orders_df.head())

#Retrieves only orders that were updated on or after October 1, 2026. 
#This demonstrates how an ETL pipeline can extract new or changed records without repeatedly extracting everything.

Incremental orders extracted: 8593


,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at
0,166585,567,2026-09-24,Home,3,80021.99,delivered,card,2026-10-01 00:04:36
1,188311,1231,2026-09-28,Electronics,2,32242.92,delivered,wallet,2026-10-01 00:06:01
2,218039,245,2026-09-25,Books,1,1057.99,delivered,cash_on_delivery,2026-10-01 00:12:26
3,261988,1626,2026-09-09,Beauty,5,9547.40,returned,card,2026-10-01 00:13:40
4,168034,956,2026-09-30,Books,2,6786.90,pending,wallet,2026-10-01 00:13:49


In [24]:
# Extract MongoDB orders

# Exclude MongoDB's internal _id field
mongo_documents = list(
    mongo_collection.find({}, {"_id": 0})
)

print("MongoDB documents extracted:", len(mongo_documents))

# Save raw documents as JSON
mongo_file = RAW_DIR / "mongo_orders.json"

with open(mongo_file, "w", encoding="utf-8") as file:
    json.dump(
        mongo_documents,
        file,
        indent=2,
        default=str
    )

print("MongoDB JSON file saved!")

#Retrieves the app orders from MongoDB and saves the original nested documents as JSON.

MongoDB documents extracted: 5000
MongoDB JSON file saved!


In [25]:
# Extract weather data

cities = {
    "Lahore": {
        "latitude": 31.55,
        "longitude": 74.34
    },
    "Karachi": {
        "latitude": 24.86,
        "longitude": 67.01
    },
    "Islamabad": {
        "latitude": 33.69,
        "longitude": 73.06
    }
}

START_DATE = "2025-01-01"
END_DATE = "2026-10-24"

weather_frames = []

for city, coordinates in cities.items():

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": coordinates["latitude"],
        "longitude": coordinates["longitude"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": "temperature_2m_max,precipitation_sum",
        "timezone": "Asia/Karachi"
    }

    response = requests.get(
        url,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    weather_data = response.json()

    # Save original weather response as JSON
    with open(
        RAW_DIR / f"weather_{city.lower()}.json",
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            weather_data,
            file,
            indent=2
        )

    # Convert daily weather data to a DataFrame
    daily_data = weather_data["daily"]

    city_weather = pd.DataFrame({
        "date": daily_data["time"],
        "temperature_2m_max": daily_data["temperature_2m_max"],
        "precipitation_sum": daily_data["precipitation_sum"]
    })

    city_weather["city"] = city

    weather_frames.append(city_weather)

    print(
        f"{city}: {len(city_weather)} daily weather records extracted"
    )

Lahore: 662 daily weather records extracted
Karachi: 662 daily weather records extracted
Islamabad: 662 daily weather records extracted


In [26]:
# Combine and save weather data

weather_df = pd.concat(
    weather_frames,
    ignore_index=True
)

weather_df.to_csv(
    RAW_DIR / "weather_daily.csv",
    index=False
)

print("Total weather records:", len(weather_df))

display(weather_df.head())

Total weather records: 1986


,date,temperature_2m_max,precipitation_sum,city
0,2025-01-01,16.3,0.0,Lahore
1,2025-01-02,17.2,0.0,Lahore
2,2025-01-03,17.4,0.0,Lahore
3,2025-01-04,18.8,0.0,Lahore
4,2025-01-05,15.9,36.6,Lahore


In [27]:
# Final Summary

print("========== EXTRACTION SUMMARY ==========")

for file in sorted(RAW_DIR.iterdir()):

    if file.is_file():
        print(f"{file.name}: {file.stat().st_size:,} bytes")

print("\nExtraction completed successfully!")

client.close()
engine.dispose()

========== EXTRACTION SUMMARY ==========
mongo_orders.json: 2,108,826 bytes
postgres_customers.csv: 95,357 bytes
postgres_incremental_orders.csv: 700,967 bytes
postgres_orders.csv: 16,301,655 bytes
weather_daily.csv: 58,421 bytes
weather_islamabad.json: 30,960 bytes
weather_karachi.json: 30,914 bytes
weather_lahore.json: 30,954 bytes

Extraction completed successfully!
